In [1]:
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt

In [2]:

data1 = loadmat('afdb_1.mat')
print(data1.keys())
rr = data1['rr'].flatten()
target = data1['targetsRR'].flatten()

# print(rr.shape)
# print(rr)
# 
# plt.figure()
# plt.plot(rr.flatten())
# plt.title("RR Intervals")
# plt.xlabel("Beats")
# plt.ylabel("RR Interval")
# plt.show()
# 

dict_keys(['__header__', '__version__', '__globals__', 'Fs', 'recordName', 'targetsQRS', 'targetsRR', 'qrs', 'rr', 'afBounds'])


In [19]:
def evaluate(target, detect):
    TP = np.sum((target == 1) & detect == 1)
    TN = np.sum((target == 0) & detect == 0)
    FP = np.sum((target == 0) & detect == 1)
    FN = np.sum((target == 1) & detect == 0)

    sensitivity = TP / (TP + FN)
    specificity = TN / (TN + FP)

    return sensitivity, specificity


In [20]:
window_size = 10
threshold = 0.022
rmssd_values = []
detectRR = []
for i in range(len(rr)-window_size+1):
    window = rr[i:i+10]
    diff = np.diff(window)
    rmssd = np.sqrt(np.mean(diff**2))
    rmssd_values.append(rmssd)

    if rmssd > threshold:
        detectRR.append(1)
    else:
        detectRR.append(0)

detectRR = np.array(detectRR)
rmssd_values = np.array(rmssd_values)

target_window = target[window_size -1:]

sensitivity, specificity = evaluate(target_window, detectRR)

print(sensitivity)
print(specificity)
# plt.figure()
# plt.plot(rmssd_values)
# plt.title("rmssd")
# plt.xlabel("DIfference")
# plt.ylabel("RR Interval")
# plt.show()
#



0.008253984038982672
0.48780548442433436


In [22]:
# Check the RMSSD range
print("Min RMSSD:", np.min(rmssd_values))
print("Max RMSSD:", np.max(rmssd_values))
print("Mean RMSSD:", np.mean(rmssd_values))
print("Median RMSSD:", np.median(rmssd_values))

# Use a data-driven threshold instead of 0.08
threshold = np.median(rmssd_values)
print("Chosen threshold:", threshold)

# Recompute detection
detectRR = (rmssd_values > threshold).astype(int)

# Evaluate
sensitivity, specificity = evaluate(target_window, detectRR)

print("Sensitivity:", sensitivity)
print("Specificity:", specificity)

# Optional: show how many windows are classified as AF
print("Number of AF detections:", np.sum(detectRR))
print("Total windows:", len(detectRR))

Min RMSSD: 0.0013333333333333346
Max RMSSD: 0.5808254662307966
Mean RMSSD: 0.040830089699143965
Median RMSSD: 0.022627416997969538
Chosen threshold: 0.022627416997969538
Sensitivity: 0.008253984038982672
Specificity: 0.5086641971011611
Number of AF detections: 20095
Total windows: 40223
